Groundwater | Transport Modeling Track

# Step 1: Model Goal – Defining the Transport Model Problem

> **The 10 steps at a glance:** **1-Goal** → 2-Perceptual → 3-MODFLOW → 4-Build → 5-Calibrate → 6-Validate → 7-Sensitivity → 8-Apply → 9-Document → 10-Communicate

**The journey:** You already know how to define goals for a flow model. Transport modeling adds a new question – not just *where does the water go?* but *what does it carry along, and where does that end up?*

In this track we follow a **conservative tracer** – an idealised dissolved substance that travels with the water without sorbing, decaying, or reacting – as it moves through the Limmat Valley aquifer between a real pair of **groundwater heat-exchange (GWHE) wells**: an *injection–abstraction doublet*. This notebook builds directly on the calibrated flow model from the flow track.

| **Core content** | ~10 minutes |
|:--|:--|
| **Optional: Exercises & real-world context** | +5 minutes |

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** clear objectives for a conservative-solute transport model on the Limmat Valley aquifer
2. **Frame** the motivating question for a GWHE doublet: does an injected tracer **short-circuit** back to the abstraction well, and how much?
3. **Identify** what additional data and parameters transport modeling requires beyond a flow model
4. **Explain** why a calibrated flow model is the essential foundation for any transport simulation

> ⚠️ **You will MODIFY a provided model — not build one**
>
> In your project you receive a working, pre-built transport model of a real Limmat GWHE doublet from the instructor. You **modify** it (source concentration, dispersivity, scenario) and interpret the results. You do **not** build the grid, refine it, or set up the packages from scratch. These notebooks show you *how the provided model was made and why*, so you can judge and adapt it.

## Prerequisites

Before starting this notebook, you should have:
- **Completed the Flow Modeling Track** – the transport model is built on top of the calibrated flow model
- Understanding of advection and dispersion concepts from the theory lectures
- Familiarity with the Limmat Valley case study and its wells from the flow track

## Introduction

A flow model tells us where water moves and how fast. A transport model asks the next question: **what travels with the water, and where does it go?**

Here that "what" is a **conservative tracer**: a dissolved substance that moves with the groundwater while we deliberately ignore sorption, decay, and reactions. It is the simplest, most transparent transport problem – advection carries the tracer along with the flow, while dispersion and diffusion spread it out – and it is exactly the case the course scopes for your project ("conservative tracer scenarios").

**The scenario.** Scattered through the Limmat Valley are **groundwater heat-exchange (GWHE) doublets** – paired wells that pump groundwater up, use it for building heating or cooling, and reinject the used water nearby. They are *designed so that the reinjected water does not flow straight back to the abstraction well* – that would short-circuit the system and degrade its performance. That design intent gives us a sharp transport question:

> 🎯 **Motivating question:** If we release a conservative tracer at the **injection** well of a real Limmat doublet, does it **short-circuit back to the abstraction well** – and if so, how much of it, and how fast?

You will answer this by **modifying a provided model** and reading the tracer breakthrough at the abstraction well.

In this notebook we define the goals and scope of that transport model. We already set the flow model's objectives in the flow track; here we focus on what changes when transport enters the picture:
- **What new questions** can we answer?
- **Where** and at what temporal and spatial scale?
- **What additional data** do we need?
- **Who else** cares about the results?

---

## What Do We Need From This Model?

> 💡 **Our Transport Model Requirements**
>
> We require a **conservative-solute transport model** of a Limmat Valley GWHE doublet to:
>
> 1. **Track a conservative tracer** released at the injection well as it is advected and dispersed through the aquifer
> 2. **Quantify breakthrough at the abstraction well** – whether and how much of the injected tracer short-circuits back, and when it arrives
> 3. Build on our existing **calibrated flow model** using **open-source tools** (MODFLOW 6 GWT)
>
> The model should be **fast enough for interactive exploration** (each breakthrough simulation completes in ≤10 minutes).

---

## 🎯 Mission Objectives – What Are We Solving?

The flow model answered: *where does the water go and how much?* Transport modeling opens up a different set of questions.

### Primary Objectives — Conservative Solute Transport

Our Limmat Valley doublet transport model should be able to:

1. **Simulate conservative-tracer migration** from the injection well through the aquifer
2. **Predict breakthrough (arrival time and concentration)** at the abstraction well – the short-circuiting question
3. **Delineate the capture zone** of the abstraction well and relate it to the flow field
4. **Test sensitivity** to the transport parameters we are least sure of – dispersivity and effective porosity

**Why a conservative tracer first?** It isolates the transport processes that the flow field controls – advection and dispersion – without the added uncertainty of reactions. Once you can read a conservative breakthrough curve, sorption and decay are incremental additions (kept optional in this course).

<details>
<summary><strong>Optional: these are heat-exchange wells — why model a solute?</strong></summary>

The doublets in this study are **real groundwater heat-exchange (GWHE) installations**: they extract groundwater, draw heat from it (or dump heat into it) for building heating and cooling, and reinject the temperature-altered water. Their genuine engineering concern is a *thermal* one – does the reinjected warm/cold water recirculate and degrade the source temperature?

We study the **conservative-solute analogue** of that same recirculation question instead, because:

- it is the case the course scopes (conservative tracer scenarios), and it maps cleanly onto a 1D analytical benchmark later in the track;
- the short-circuiting geometry is identical – a tracer injected at the reinjection well stands in for any conservative signature carried in the returned water;
- heat carries extra physics (storage in, and conduction through, the solid matrix, which retards a thermal front relative to the water) that we deliberately set aside here.

So: same wells, same recirculation question, a simpler and more general tracer. The heat-transport packages (MODFLOW 6 **GWE**) are the thermal counterpart of the solute packages (**GWT**) we use here.
</details>

### Educational Objectives

As a teaching tool, the transport model should also:

- Illustrate how the **flow field controls transport** (the doublet's forced gradient sets the tracer path)
- Show the roles of **advection, dispersion, and diffusion**
- Demonstrate the difference between **conservative and reactive** transport
- Highlight the importance of **numerical stability** (Peclet and Courant criteria)

<details>
<summary><strong>Optional: Exercise – Reasoning about short-circuiting</strong></summary>

> ✏️ **Exercise: Will the tracer come back?**
>
> Before we model anything, reason qualitatively about the motivating question:
>
> 1. The doublet is *designed* so that reinjected water does not flow straight back to the abstraction well. Describe two paths a tracer particle could take from the injection well – one that returns to the abstraction well, and one that escapes downgradient. What controls which one dominates?
> 2. Dispersivity spreads the tracer out around the average flow path. Would *increasing* dispersivity make a short-circuit signal at the abstraction well easier or harder to detect? Why?
> 3. Effective porosity sets the seepage velocity ($v = q / n_e$). If you halved the effective porosity, what would happen to the tracer's arrival time at the abstraction well?
>
> **Hint:** Advection follows the forced-gradient flow field between the two wells; dispersion smears the front around it. Arrival *time* scales with travel distance divided by seepage velocity.

</details>

---

## 🗺️ Setting the Scene – Where and When?

We use the same Limmat Valley domain and the same flow grid as the flow model, but transport places different demands on resolution and time.

### Spatial Scale

| Dimension | Value | Rationale |
|-----------|-------|-----------|
| **Domain extent** | Same as flow model | Full valley, focused on the doublet and its capture zone |
| **Grid resolution** | Inherited flow grid (~50 m cells), locally refined near the doublet | Sharp concentration gradients need finer cells than flow — see the note below |
| **Vertical layers** | 1 | Single layer — prioritises fast run times for interactive exploration |

> ⚠️ **A grid that is fine enough for flow is often too coarse for transport.** On the ~50 m flow grid, the tracer front is artificially smeared (the *grid Peclet* number is too high). The provided model is therefore **refined near the doublet** so the breakthrough is trustworthy. You will see exactly how this is diagnosed and fixed in **[04t](04t_model_implementation.ipynb)** — here it is enough to know that resolution must be checked against the *transport*, not just the flow.

### Temporal Scale

| Aspect | Choice | Rationale |
|--------|--------|-----------|
| **Simulation type** | Transient | The tracer plume evolves over time, even on a steady flow field |
| **Time step** | Small near the wells | Must satisfy the Courant criterion where velocities are highest (near the doublet) |
| **Simulation period** | Long enough for breakthrough | Run until the tracer either arrives at the abstraction well or clearly escapes |

> **Key insight:** Transport simulations are almost always **transient** – even when the underlying flow is steady-state. The tracer concentration at the abstraction well after 1 year looks very different from after 10 years.

---

## Essential Data – What Do We Need?

The flow model already supplies the velocity field. Conservative-solute transport adds its own, shorter, list of requirements.

### Data for Conservative-Solute Transport

| Category | Data Type | Source | Availability |
|----------|-----------|--------|--------------|
| **Porosity** | Effective porosity *n*<sub>e</sub> | Literature, lab tests | ⚠️ Estimated |
| **Dispersivity** | Longitudinal *α*<sub>L</sub>, transverse *α*<sub>T</sub> | Tracer tests, literature | ⚠️ Uncertain |
| **Diffusion** | Molecular diffusion *D*<sub>m</sub> | Literature | Good |
| **Source term** | Injection-well location, tracer concentration, flow rate | Doublet design | Known |
| **Background** | Ambient (initial) concentration | Definition | Set to zero |
| **Receptor** | Abstraction-well location and rate | Doublet design | Known |

> **Working defaults used in this track** (gravel aquifer): effective porosity *n*<sub>e</sub> = 0.20; longitudinal dispersivity *α*<sub>L</sub> = 10 m; transverse dispersivity *α*<sub>T</sub> = 1 m (= *α*<sub>L</sub>/10); molecular diffusion *D*<sub>m</sub> ≈ 8.6 × 10⁻⁵ m²/d (≈ 1 × 10⁻⁹ m²/s). You will revisit and stress-test *α*<sub>L</sub> as a sensitivity exercise later in the track.

> **Key terms:**
> - *Effective porosity (n<sub>e</sub>)*: The fraction of pore space through which water (and the tracer) actually flows. Controls transport velocity via *v* = *q* / *n*<sub>e</sub>.
> - *Dispersivity (α)*: A length scale that governs mechanical mixing. Notoriously scale-dependent – values measured in lab columns don't apply at field scale.
> - *Conservative tracer*: A substance that moves with the water without sorbing, decaying, or reacting (retardation factor *R* ≈ 1).

### Data Quality Considerations

- **The source term often dominates the uncertainty**: what is released, at what concentration and rate. For our doublet this is *known by design* (the injection well's flow rate and the tracer concentration we assign), which is exactly what makes a doublet such a clean transport experiment.
- **Dispersivity is scale-dependent**: it grows with the distance the tracer travels, because longer paths sample more heterogeneity. A lab value of ~1 cm can become tens of metres at field scale – one of the most uncertain parameters in solute transport.
- **Effective porosity is rarely measured directly** and sets the seepage velocity, so it directly controls arrival time.
- **Observations**: real concentration data are typically sparse and expensive; in this teaching study the "observation" is the simulated breakthrough at the abstraction well.

### Comprehension Check

Before moving on, make sure you can answer these:

> **1. For a conservative-tracer study of a GWHE doublet, which two inputs usually carry the most uncertainty – and why?**
>
> **2. Why does this course have you *modify a provided* transport model rather than build one from scratch?**

<details>
<summary>Click to check your answer</summary>

**1. The source term and dispersivity.**
- The **source term** (what is released, at what concentration and rate) is the dominant uncertainty in most real solute studies – although in our doublet it is fixed *by design*, which is what makes the breakthrough question well-posed.
- **Dispersivity** is *scale-dependent*: it grows with travel distance because longer paths sample more heterogeneity, so a lab value of ~1 cm can become tens of metres in the field.

Effective porosity is a close third, because it sets the seepage velocity and hence the arrival time. These are the parameters you will deliberately stress-test, because the flow field – and therefore advection – is already constrained by the calibrated flow model.

**2. Because building and refining a transport grid correctly is instructor-time work.** A grid that is adequate for flow is *not* adequate for transport (the grid-Peclet problem, shown in 04t), and refining it reliably near the wells is fiddly. Handing you a pre-built, pre-refined model lets you spend your time on the modeling *decisions and judgment* – source, dispersivity, scenario, and interpreting the breakthrough – which is where the learning is.
</details>

---

## 👥 Key Players – Who Cares About the Results?

Transport results often carry higher stakes than flow-only results – contamination affects human health, and well-field operators care whether their reinjected water comes back.

### Primary Users

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Students** | Learning transport concepts | Clear breakthrough curves, interactive scenarios |
| **Instructors** | Teaching advection–dispersion | An illustrative, fast-running doublet example |

<details>
<summary><strong>Optional: Real-world stakeholders</strong></summary>

### In a Real-World Context

| Stakeholder | Interest | Model Requirements |
|-------------|----------|-------------------|
| **Regulators** | Compliance with concentration limits | Conservative predictions, uncertainty bounds |
| **Site owners** | Contamination liability, remediation costs | Plume delineation, arrival-time estimates |
| **Water utilities** | Wellhead protection, safe yield | Capture-zone analysis, early warning |
| **GWHE / geothermal operators** | System efficiency, avoiding recirculation | Whether and how fast reinjected water short-circuits to the abstraction well |

</details>

---

## Summary: Transport Model Goal Definition

> **🎯 Mission**
>
> **Track a conservative tracer** from the injection well of a real Limmat GWHE doublet and **read its breakthrough at the abstraction well** – does it short-circuit, how much, and when?
> - Primary: arrival time and concentration at the abstraction well; the abstraction-well capture zone
> - Sensitivity: dispersivity (*α*<sub>L</sub>) and effective porosity
>
> **🗺️ Setting the Scene**
>
> - Spatial: same domain and flow grid, locally refined near the doublet (a flow-adequate grid is too coarse for transport – see 04t)
> - Temporal: transient simulation on a steady flow field, run until breakthrough or clear escape
>
> **📊 Essential Intel**
>
> - Foundation: the calibrated flow model (supplies the velocity field)
> - Add: effective porosity, dispersivity (*α*<sub>L</sub> = 10 m, *α*<sub>T</sub> = 1 m), and the injection-well source term
>
> **🔑 Key Message**
>
> A conservative tracer isolates the transport physics the flow field controls; the source term and dispersivity carry the uncertainty.
>
> **👥 Key Players**
>
> - Primary: Students and instructors (educational use)
> - Real-world: Regulators, site owners, water utilities, GWHE/geothermal operators

---

## What You're Taking Forward

You now have a clear, well-posed transport goal: a **conservative tracer released at a GWHE doublet's injection well**, with the **breakthrough at the abstraction well** as the quantity of interest. The calibrated flow model supplies the velocity field; transport adds just three things – **effective porosity, dispersivity, and the source term**.

Remember the two framing decisions:
- You will **modify a provided, pre-refined model**, not build or refine one.
- The biggest uncertainties are the **source term and dispersivity** – which is exactly where your judgment and sensitivity testing belong.

---

## Next Steps

With the transport goal defined, we extend our perceptual model to include transport processes – what actually drives a tracer's movement between the doublet wells in the Limmat Valley?

**Continue to:** [02t_perceptual_model.ipynb](02t_perceptual_model.ipynb)